In [ ]:
from os import listxattr
import random
import json
import glob

# **Clase SchemaData**

Se encarga de obtener el fichero con los identificadore de los pacientes que se emplean en la etapa de entrenamiento del modelo.


In [ ]:

class SchemaData:
  global base_path
  base_path = "/content/drive/MyDrive/"

  def __init__(self, name, percent):
    '''
    @name: nombre del experimento (test paciente)
    @percent: porcentaje de datos de entrenamiento
    '''
    self.path_H = f"{base_path}{name}Control_out_resize/"
    self.path_S = f"{base_path}{name}Patients_out_resize/"
    self.percent_train = percent
    self.file_schema = f"{base_path}{name}_id_train_test_{percent}_percent.json"


  def user_by_test(self, path):
    """
    @path: ruta con las imagenes de los pacientes
    @return: lista con los identificadores de los pacientes
    """
    user_list = []
    for i in glob.glob(path+"*_TH.jpg"):
      user = i.split("/")[-1].split("-")[0]
      if user not in user_list:
        user_list.append(user)
    return user_list

  def search_percent(self, n_data):
    """
    @n_data: numero de datos totales
    @return: numero de datos de entrenamiento
    """
    return int(n_data*self.percent_train/100)

  def select_users(self, users_list, nums):
    """
    @users_list: lista con los identificadores de los pacientes
    @nums: porcentaje de datos de entrenamiento
    @return: lista de identificadores de los pacientes seleccionados aleatorios
    """
    return random.sample(users_list, nums)

  def homogeous_data(self, user_c, user_p):
    """
    @user_c: lista con los identificadores de los pacientes enfermos
    @user_p: lista con los identificadores de los pacientes sanos
    @return: numero de datos por tipo de datos y si los datos son balanceados
    """
    balance = True
    n_data = len(user_c)
    if len(user_c) != len(user_p):
      n_data = len(user_p) if len(user_c) > len(user_p) else len(user_c)
      balance = False
    return n_data, balance


  def save_schema_data(self, users_control_train, users_patients_train,
                      users_control_test, users_patients_test, balance):
    '''
    @users_control_train: lista con los ids de los pacientes enfermos para train
    @users_patients_train: lista con los ids de los pacientes sanos para train
    @users_control_test: lista con los ids de los pacientes enfermos para test
    @users_patients_test: lista con los ids de los pacientes sanos para test
    @balance: si los datos estan balanceados
    '''
    data_json = dict(id_train=dict(sick=users_patients_train,
                       healthy=users_control_train),
         id_test=dict(sick=users_patients_test,
                      healthy=users_control_test),
         balance=balance)
    json.dump(data_json, open(self.file_schema, "w"))


  def run(self):
    """
    @return None
    """
    users_control = self.user_by_test(self.path_H)
    users_patients = self.user_by_test(self.path_S)
    n_data, balance = self.homogeous_data(users_control, users_patients)
    percents_steps = self.search_percent(n_data)
    users_control_train = self.select_users(users_control, percents_steps)
    users_patients_train = self.select_users(users_patients, percents_steps)
    users_control_test = list(set(users_control) - set(users_control_train))
    users_patient_test = list(set(users_patients) - set(users_patients_train))
    self.save_schema_data(users_control_train, users_patients_train,
                     users_control_test, users_patient_test, balance)


# Ejecución

In [ ]:
for i in ["Meander", "Spiral"]
  SchemaData(i, 60).run()